# Data Fusion

This notebook merges the **Balanced On-Chain Data** with the **Selected Off-Chain Features** to create the final training dataset for XGBoost.

## Inputs
- **On-Chain**: `balanced_label_days.parquet` (Address-Level, Balanced)
- **Off-Chain**: `market_daily_cleaned.parquet` (Global Daily, Top 4 Features)
- **Off-Chain**: `reddit_daily_cleaned.parquet` (Global Daily, Top 4 Features)

## Output
- `final_train_data.parquet`

## Final Feature Set
    # On-Chain (4)
    'normal_total_cnt',
    'uniq_peers_cnt',
    'burst_max_tx_5m',
    'normal_sent_cnt',
    # Reddit (3)
    'reddit_fraud_mention_ratio',
    'reddit_total_activity',
    'reddit_avg_sentiment',
    # Market (3)
    'eth_volatility_7d',
    'eth_daily_return',
    'eth_intraday_volatility'


In [1]:
import pandas as pd
import os

onchain_path = '../data/processed/chain/balanced_label_days.parquet'
market_path = '../data/processed/market/market_daily_cleaned.parquet'
reddit_path = '../data/processed/reddit/reddit_daily_cleaned.parquet'
output_dir = '../data/processed/final'
os.makedirs(output_dir, exist_ok=True)

In [2]:
chain_df = pd.read_parquet(onchain_path)
print(f"On-Chain Shape: {chain_df.shape}")

market_df = pd.read_parquet(market_path)
print(f"Off-Chain Shape: {market_df.shape}")

reddit_df = pd.read_parquet(reddit_path)
print(f"Off-Chain Shape: {reddit_df.shape}")

chain_df['day'] = pd.to_datetime(chain_df['day']).dt.date
market_df['day'] = pd.to_datetime(market_df['day']).dt.date
reddit_df['day'] = pd.to_datetime(reddit_df['day']).dt.date

On-Chain Shape: (66557, 14)
Off-Chain Shape: (1766, 4)
Off-Chain Shape: (1766, 4)


In [3]:
offchain_df = pd.merge(market_df, reddit_df, on='day', how='outer')

offchain_df = offchain_df.sort_values('day').fillna(0)

offchain_output_para = os.path.join(output_dir, 'offchain_daily.parquet')
offchain_df.to_parquet(offchain_output_para, index=False)

offchain_output_csv = os.path.join(output_dir, 'offchain_daily.csv')
offchain_df.to_csv(offchain_output_csv, index=False)

print(f"Saved off-chain dataset")
print(f"Off-chain Shape: {offchain_df.shape}")

Saved off-chain dataset
Off-chain Shape: (1766, 7)


In [4]:
merged_df = pd.merge(chain_df, offchain_df, on='day', how='left')

merged_df = merged_df.fillna(0)

print(f"Full merged shape: {merged_df.shape}")


Full merged shape: (66557, 20)


In [5]:
final_features = [
    # Identity & Target
    'address', 'day', 'is_anomalous',

    # On-Chain (4)
    'normal_total_cnt',
    'uniq_peers_cnt',
    'burst_max_tx_5m',
    'normal_sent_cnt',
    # Reddit (4)
    'reddit_fraud_mention_ratio',
    'reddit_total_activity',
    'reddit_avg_sentiment',
    # Market (4)
    'eth_volatility_7d',
    'eth_daily_return',
    'eth_intraday_volatility'
]

train_df = merged_df[final_features].copy()

print(f"Final Training Set Shape: {train_df.shape}")
print("\nClass Distribution:")
print(train_df['is_anomalous'].value_counts())

train_df.head()

Final Training Set Shape: (66557, 13)

Class Distribution:
is_anomalous
0    33289
1    33268
Name: count, dtype: int64


,address,day,is_anomalous,normal_total_cnt,uniq_peers_cnt,burst_max_tx_5m,normal_sent_cnt,reddit_fraud_mention_ratio,reddit_total_activity,reddit_avg_sentiment,eth_volatility_7d,eth_daily_return,eth_intraday_volatility
0,0xd624d046edbdef805c5e4140dce5fb5ec1b39a3c,2017-03-15,1,2,2,2,1,0.077586,580,0.371972,0.080752,0.223395,0.231198
1,0xd624d046edbdef805c5e4140dce5fb5ec1b39a3c,2017-03-16,1,3,2,2,1,0.081192,973,0.286423,0.108345,0.322054,0.334089
2,0xc859d7753a0295d8b88be34014a4956e15a2b745,2017-03-28,0,2,2,2,1,0.069388,735,0.338754,0.086855,0.022309,0.036435
3,0x04786aada9deea2150deab7b3b8911c309f5ed90,2017-04-03,1,4,4,2,4,0.066798,509,0.268998,0.046275,-0.090101,0.110843
4,0xbad41e5412ebaf22b8ec127521060fd9fa29bbdd,2017-04-14,0,2,2,2,1,0.073171,369,0.324839,0.050267,-0.052675,0.065790


In [6]:
output_file_para = os.path.join(output_dir, 'final_train_data.parquet')
train_df.to_parquet(output_file_para, index=False)

output_file_csv = os.path.join(output_dir, 'final_train_data.csv')
train_df.to_csv(output_file_csv, index=False)
print(f"\n Saved Final Training Data")


 Saved Final Training Data
